In [8]:
from typing import Annotated

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

from langchain_core.messages import trim_messages
from langchain_core.messages.utils import count_tokens_approximately

from langchain_ollama import ChatOllama

In [9]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    base_url="http://172.31.0.1:11434"
)

In [10]:
def call_model(state: MessagesState):

    messages = state["messages"]

    trimmed_messages = trim_messages(
        messages,
        max_tokens=100,
        strategy="last",
        token_counter=count_tokens_approximately,
        include_system=True,
        allow_partial=False
    )

    response = llm.invoke(trimmed_messages)

    return {
        "messages": [response]
    }

In [11]:
builder = StateGraph(MessagesState)

builder.add_node("call_model", call_model)

builder.add_edge(START, "call_model")
builder.add_edge("call_model", END)

In [12]:
memory = InMemorySaver()

graph = builder.compile(
    checkpointer=memory
)

In [13]:
config = {
    "configurable": {
        "thread_id": "1"
    }
}